In [2]:
with open("input.txt","r") as file:
    text = file.read()



In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

#Code for creating the encoder and decoder
stoi = {ch:i for i ,ch in enumerate(chars)}
itos = {i:ch for i ,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("test"))

[58, 43, 57, 58]


In [11]:
import torch
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape)

#Split into validation and train set
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

torch.Size([1115393])


In [ ]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [ ]:
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets=None):

        logits = self.token_embedding_table(idx)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T,C)

            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)

        
        return logits, loss
    
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next),dim=1)
        return idx
    
m = BigramLanguageModel(vocab_size)
xb,yb = get_batch("train")
out,loss = m(xb,yb)
print(out.shape)
print(loss)

print(decodem.generate(xb,10))

torch.Size([32, 65])
tensor(4.7843, grad_fn=<NllLossBackward0>)


KeyError: tensor([ 0, 20, 53, 61,  1, 57, 39, 63, 50,  7, 29, 35, 49, 58, 36, 53, 24,  4])